In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,datediff,coalesce
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_settlement_exposure_process(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "SettlementExposureId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, SettlementExposureKey string, InitialSettlementKey int,"
        #     "PreviouSettlementKey int, SettlemenKey int, ExposureKey string, AgreementKey string, PartyKey string, CDI string,"
        #     "CaseCode int, DebtAmount decimal(19,4), PrincipalProposedAmount decimal(19,2), ProposedAmount decimal(19,4),"
        #     "RemainAmount decimal(19,4), SettlementAmount decimal(19,4), ExpectedSettlementAmount decimal(19,4),"
        #     "DiscountAmount decimal(19,4), DiscountPercentage decimal(19,4), BalloonAmount decimal(19,2),"
        #     "BucketCode int, Bucket int, SettlementTypeCode string, SettlementType string, SettlementStatusCode string,"
        #     "SettlementStatus string, CreatedDate timestamp, SubmissionDate timestamp, ApprovalDate date,"
        #     "ImplementationDate timestamp, ExpiredDate timestamp, CancelledDate timestamp, RejectedDate date,"
        #     "OverdueDate date, FirstInstalmentDate date, NextInstalmentDate date, PastInstalmentAmount decimal(19,4),"
        #     "PastInstalmentNumber int, NextInstalmentAmount decimal(19,4), NumOfaccounts int, NumOfinstalments int,"
        #     "TotalPaymentAmount decimal(19,4), IsActiveFlag int, Comments string, KeptPercentage decimal(19,4),"
        #     "IsMigratedFlag boolean, MigrationComments string, PoolName string, AddedManuallyFlag boolean,"
        #     "isPrimarySettlementFlag boolean, ParticipationPercentage decimal(19,4), MaturityPercentage decimal(19,4),"
        #     "MutiplyforLCY decimal(19,4), OtherExpense1 decimal(19,4), OtherExpense2 decimal(19,4), OtherExpense3 decimal(19,4),"
        #     "LastUpdatedDate timestamp, CancellationReasonCode int, CancellationReason string, DiasRFCode string,"
        #     "CtrlMovedFlag boolean,ContractNumber string,CaseFrozenInterestAmtLcy decimal(19,4),CaseProvisionAmtLcy decimal(19,4),"
        #     "MvhoCaseFrozenInterestAmt decimal(19,4),MvhoCaseWriteOffAmt  decimal(19,4), MvhoCaseProvisionAmt  decimal(19,4),"
        #     "SettlementLastInstallmentDate  date, CaseWriteOffAmtLcy  decimal(19,4), StartingOutstandingBalance decimal(19,4), StartingDebtAmount decimal(19,4),"
        #     "StartingFrozen decimal(19,4), StartingWriteOffAmount decimal(19,4), StartingCurrentBucket  int, StartingCaseDenounceStatus string,StartingHostAccountStatus string,"
        #     "SepsIsDownPayment boolean,SepsInstallmentAmt  decimal(19,4),AgencyCode string,BusinessDate date"
        # )
        
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.SettlementExposure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "process",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "vendor_code_qualco",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== settlement ==================

        settlement_extra_filters = None

        if settlement_extra_filters is not None:
            settlement_extra_filters.append(date_filter)
            settlement_final_filters = " AND ".join(settlement_extra_filters)
        else:
            settlement_final_filters = date_filter

        settlement_selected_fields = [
            "settlement_code",
            "vendor_code",
            "initial_settlement_code",
            "prv_settlement_code",
            "debt_amount",
            "principal_proposed_amount",
            "proposed_amount",
            "remain_amount",
            "settlement_amount",
            "expected_settle_amount",
            "discount_amount",
            "balloon_amount",
            "settlement_bucket",
            "settlement_type",
            "status",
            "created",
            "submission_dt",
            "approval_dt",
            "implementation_dt",
            "expiry_dt",
            "cancelled_dt",
            "rejected_dt",
            "overdue_date",
            "first_instalment_date",
            "next_instalment_date",
            "next_instalment_amount",
            "num_of_accounts",
            "num_of_instalments",
            "tot_payment_amount",
            "is_active",
            "comments",
            "kept_percentage",
            "is_migrated",
            "migration_comments",
            "pool_name",
            "businessdate",
        ]

        settlement_prev = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "process_settlement",
            settlement_final_filters,
            settlement_selected_fields,
        )

        # ================== settlement_loan ==================

        settlement_loan_extra_filters = None

        if settlement_loan_extra_filters is not None:
            settlement_loan_extra_filters.append(date_filter)
            settlement_loan_final_filters = " AND ".join(settlement_loan_extra_filters)
        else:
            settlement_loan_final_filters = date_filter

        settlement_loan_selected_fields = [
            "account_number",
            "cdi",
            "case_code",
            "added_manually_flg",
            "isprimarysettlement",
            "participation_perc",
            "settlement_code",
            "businessdate",
        ]

        settlement_loan = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "process_settlement_loan",
            settlement_loan_final_filters,
            settlement_loan_selected_fields,
        )

        settlement = settlement_prev.withColumnRenamed("settlement_code", "sete_code")

        # Join conditions
        settlement_settlement_loan_join_condition = (
            settlement["sete_code"] == settlement_loan["settlement_code"]
        )

        settlement_portfolio_join_condition = (
            settlement["vendor_code"] == portfolio["vendor_code_qualco"]
        )

        past_installments = (
        settlement
        .join(settlement_loan, settlement_settlement_loan_join_condition, "left")
        .filter(settlement["next_instalment_date"] < settlement["businessdate"])
        .groupBy(settlement_loan["account_number"])
        .agg(F.sum(settlement["num_of_instalments"]).alias("PastNumberofInstallments"))
        .withColumnRenamed("account_number", "settl_account_number")
        )

        target_table_process = (
        settlement
        .join(settlement_loan, settlement_settlement_loan_join_condition, "left")
        .join(portfolio, settlement_portfolio_join_condition, "left")
        .join(
            past_installments,
            on=settlement_loan["account_number"] == past_installments["settl_account_number"],
            how="left",
    )
)


        # HashKeys
        target_table_process_SettlementId_primary_key_fields = [
            settlement.sete_code,
            settlement_loan.account_number,
            lit("process"),
        ]

        # Target table (hash keys and calculations)
        target_table_process = target_table_process.withColumn(
            "SettlementExposureId",
            sha2(
                concat_ws(";", *target_table_process_SettlementId_primary_key_fields),
                256,
            ),
        ).withColumn(
            "SettlementExposureKey",
            concat_ws(
                ";",
                settlement.sete_code,
                settlement_loan.account_number,
            ),
        ).withColumn(
            "PastNumberofInstallments",
            F.coalesce(F.col("PastNumberofInstallments"), F.lit(0)),
        ).withColumn(
            "DaysPastDue",
            datediff(settlement.businessdate, coalesce(settlement.overdue_date, settlement.businessdate)),
        )


        # ================== Selection of Final Columns ==================

        result_df_process = target_table_process.select(
            target_table_process.SettlementExposureId.alias("SettlementExposureId"),
            lit("process").cast("string").alias("SourceSystem"),
            settlement.vendor_code.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            target_table_process.SettlementExposureKey.alias("SettlementExposureKey"),
            settlement.initial_settlement_code.cast("int").alias(
                "InitialSettlementKey"
            ),
            settlement.prv_settlement_code.cast("int").alias("PreviouSettlementKey"),
            settlement.sete_code.cast("int").alias("SettlementKey"),
            settlement_loan.account_number.cast("string").alias("ExposureKey"),
            lit(None).cast("string").alias("AgreementKey"),
            lit(None).cast("string").alias("PartyKey"),
            settlement_loan.cdi.cast("string").alias("CDI"),
            settlement_loan.case_code.cast("int").alias("CaseCode"),
            settlement.debt_amount.cast("decimal(19,4)").alias("DebtAmount"),
            settlement.principal_proposed_amount.cast("decimal(19,2)").alias(
                "PrincipalProposedAmount"
            ),
            settlement.proposed_amount.cast("decimal(19,4)").alias("ProposedAmount"),
            settlement.remain_amount.cast("decimal(19,4)").alias("RemainAmount"),
            settlement.settlement_amount.cast("decimal(19,4)").alias(
                "SettlementAmount"
            ),
            settlement.expected_settle_amount.cast("decimal(19,4)").alias(
                "ExpectedSettlementAmount"
            ),
            settlement.discount_amount.cast("decimal(19,4)").alias("DiscountAmount"),
            lit(None).cast("decimal(19,4)").alias("DiscountPercentage"),
            settlement.balloon_amount.cast("decimal(19,2)").alias("BalloonAmount"),
            settlement.settlement_bucket.cast("int").alias("BucketCode"),
            settlement.settlement_bucket.cast("int").alias("Bucket"),
            settlement.settlement_type.cast("string").alias("SettlementTypeCode"),
            settlement.settlement_type.cast("string").alias("SettlementType"),
            settlement.status.cast("string").alias("SettlementStatusCode"),
            settlement.status.cast("string").alias("SettlementStatus"),
            settlement.created.cast("timestamp").alias("CreatedDate"),
            settlement.submission_dt.cast("timestamp").alias("SubmissionDate"),
            settlement.approval_dt.cast("date").alias("ApprovalDate"),
            settlement.implementation_dt.cast("timestamp").alias("ImplementationDate"),
            settlement.expiry_dt.cast("timestamp").alias("ExpiredDate"),
            settlement.cancelled_dt.cast("timestamp").alias("CancelledDate"),
            settlement.rejected_dt.cast("date").alias("RejectedDate"),
            settlement.overdue_date.cast("date").alias("OverdueDate"),
            settlement.first_instalment_date.cast("date").alias("FirstInstalmentDate"),
            settlement.next_instalment_date.cast("date").alias("NextInstalmentDate"),
            lit(None).cast("decimal(19,4)").alias("PastInstalmentAmount"),
            target_table_process.PastNumberofInstallments.cast("int").alias("PastInstalmentNumber"),
            settlement.next_instalment_amount.cast("decimal(19,4)").alias(
                "NextInstalmentAmount"
            ),
            settlement.num_of_accounts.cast("int").alias("NumOfaccounts"),
            settlement.num_of_instalments.cast("int").alias("NumOfinstalments"),
            settlement.tot_payment_amount.cast("decimal(19,4)").alias(
                "TotalPaymentAmount"
            ),
            settlement.is_active.cast("int").alias("IsActiveFlag"),
            settlement.comments.cast("string").alias("Comments"),
            settlement.kept_percentage.cast("decimal(19,4)").alias("KeptPercentage"),
            settlement.is_migrated.cast("boolean").alias("IsMigratedFlag"),
            settlement.migration_comments.cast("string").alias("MigrationComments"),
            settlement.pool_name.cast("string").alias("PoolName"),
            settlement_loan.added_manually_flg.cast("boolean").alias(
                "AddedManuallyFlag"
            ),
            settlement_loan.isprimarysettlement.cast("boolean").alias(
                "isPrimarySettlementFlag"
            ),
            settlement_loan.participation_perc.cast("decimal(19,4)").alias(
                "ParticipationPercentage"
            ),
            lit(None).cast("decimal(19,4)").alias("MaturityPercentage"),
            lit(None).cast("decimal(19,4)").alias("MutiplyforLCY"),
            lit(None).cast("decimal(19,4)").alias("OtherExpense1"),
            lit(None).cast("decimal(19,4)").alias("OtherExpense2"),
            lit(None).cast("decimal(19,4)").alias("OtherExpense3"),
            lit(None).cast("timestamp").alias("LastUpdatedDate"),
            lit(None).cast("int").alias("CancellationReasonCode"),
            lit(None).cast("string").alias("CancellationReason"),
            lit(None).cast("string").alias("DiasRFCode"),
            settlement.businessdate.cast("date").alias("BusinessDate")
            ,lit(None).cast("decimal(19,4)").alias("MvhoCaseFrozenInterestAmt")
            ,lit(None).cast("decimal(19,4)").alias("MvhoCaseWriteOffAmt")
            ,lit(None).cast("decimal(19,4)").alias("MvhoCaseProvisionAmt")
            ,lit(None).cast("date").cast("date").alias("SettlementLastInstallmentDate")
            ,lit(None).cast("decimal(19,4)").alias("CaseWriteOffAmtLcy")
            ,lit(None).cast("decimal(19,4)").alias("StartingOutstandingBalance")
            ,lit(None).cast("decimal(19,4)").alias("StartingDebtAmount")
            ,lit(None).cast("decimal(19,4)").alias("StartingFrozen")
            ,lit(None).cast("decimal(19,4)").alias("StartingWriteOffAmount")
            ,lit(None).cast("int").alias("StartingCurrentBucket")
            ,lit(None).cast("string").alias("StartingCaseDenounceStatus")
            ,lit(None).cast("string").alias("StartingHostAccountStatus")
            ,lit(None).cast("boolean").alias("SepsIsDownPayment")
            ,lit(None).cast("decimal(19,4)").alias("SepsInstallmentAmt")
            ,lit(None).cast("string").alias("AgencyCode")
            ,lit(None).cast("boolean").alias("CtrlMovedFlag")
            ,lit(None).cast("string").alias("ContractNumber")
            ,lit(None).cast("decimal(19,4)").alias("CaseFrozenInterestAmtLcy")
            ,lit(None).cast("decimal(19,4)").alias("CaseProvisionAmtLcy")
            ,target_table_process.DaysPastDue.alias("DaysPastDue")
           
        )

        result_df_process.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_settlement_exposure_process(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)